<a href="https://colab.research.google.com/github/SarveeshK/HCL/blob/main/Unit-2/Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install Hugging Face datasets (to download the data) and keras-tuner (for auto hyperparameter tuning)
!pip install -q datasets keras-tuner

In [4]:
# Import basic data handling libraries
import numpy as np
import pandas as pd

# Import TensorFlow and dataset loader from Hugging Face
import tensorflow as tf
from datasets import load_dataset

# Import text preprocessing tools to convert words into numbers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Import Neural Network layers and KerasTuner
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, Dense, Dropout, GlobalAveragePooling1D, BatchNormalization
)
import keras_tuner as kt

# Set random seed so we get the exact same high accuracy every time we run it
tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [5]:
# Download the multiclass sentiment dataset directly from Hugging Face
print("Downloading Sp1786/multiclass-sentiment-analysis-dataset...")
dataset = load_dataset("Sp1786/multiclass-sentiment-analysis-dataset")

# Extract training text (convert to lowercase and remove extra spaces) and labels (0=Neg, 1=Neu, 2=Pos)
train_texts = [str(t).strip().lower() for t in dataset['train']['text']]
y_train = np.array(dataset['train']['label'])

# Extract validation text and labels (used to check accuracy during tuning)
val_texts = [str(t).strip().lower() for t in dataset['validation']['text']]
y_val = np.array(dataset['validation']['label'])

# Extract unseen testing text and labels (used for final accuracy check)
test_texts = [str(t).strip().lower() for t in dataset['test']['text']]
y_test = np.array(dataset['test']['label'])

# Print total number of rows in each split
print(f"Train samples      : {len(train_texts)}")
print(f"Validation samples : {len(val_texts)}")
print(f"Test samples       : {len(test_texts)}")

Train samples      : 31232
Validation samples : 5205
Test samples       : 5206


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.utils import to_categorical

# Extract top 25,000 single words and 2-word phrases (bigrams) with sublinear TF-IDF scaling
vectorizer = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Convert text splits into dense float32 feature matrices
X_train = vectorizer.fit_transform(train_texts).toarray().astype('float32')
X_val = vectorizer.transform(val_texts).toarray().astype('float32')
X_test = vectorizer.transform(test_texts).toarray().astype('float32')

# One-hot encode labels (3 classes) so we can track per-class accuracy (>80%)
y_train_oh = to_categorical(y_train, num_classes=3)
y_val_oh = to_categorical(y_val, num_classes=3)
y_test_oh = to_categorical(y_test, num_classes=3)

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)
print("X_test shape :", X_test.shape)

X_train shape: (31232, 25000)
X_val shape  : (5205, 25000)
X_test shape : (5206, 25000)


In [7]:
# Import required layers and KerasTuner directly so this cell is self-contained
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
import keras_tuner as kt

def build_dense_model(hp):
    # Predefined hyperparameter search ranges
    num_layers = hp.Int('num_dense_layers', min_value=1, max_value=2)
    units = hp.Choice('units', values=[128, 256, 512])
    dropout_rate = hp.Float('dropout', min_value=0.3, max_value=0.6, step=0.1)
    lr = hp.Choice('learning_rate', values=[1e-3, 5e-4, 1e-4])

    model = Sequential()
    model.add(tf.keras.Input(shape=(25000,)))

    # Add 1 or 2 hidden Dense layers based on tuner selection
    for _ in range(num_layers):
        model.add(Dense(units=units, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(rate=dropout_rate))

    # Output Layer: 3 classes (Negative, Neutral, Positive)
    model.add(Dense(3, activation='softmax'))

    # Compile with label smoothing to boost accuracy >80%
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.CategoricalAccuracy(name='exact_match_acc')
        ]
    )
    return model

In [8]:
import tensorflow as tf
import keras_tuner as kt

# Test 8 hyperparameter combinations and optimize for validation accuracy
tuner = kt.RandomSearch(
    build_dense_model,
    objective='val_accuracy',
    max_trials=8,
    executions_per_trial=1,
    directory='hf_tuning',
    project_name='dense_nn_high_acc',
    overwrite=True
)

# Stop early when validation loss plateaus to prevent overfitting
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=0)
]

print("Starting Automated Hyperparameter Search for Dense NN...")
tuner.search(
    X_train, y_train_oh,
    epochs=8,
    batch_size=128,
    validation_data=(X_val, y_val_oh),
    callbacks=callbacks,
    verbose=1
)

Trial 8 Complete [00h 00m 31s]
val_accuracy: 0.7700287103652954

Best val_accuracy So Far: 0.7777138948440552
Total elapsed time: 00h 05m 09s


In [9]:
import pandas as pd

# Store all tested hyperparameter combinations in a clean table
trials_data = []
for trial in tuner.oracle.get_best_trials(num_trials=8):
    p = trial.hyperparameters.values
    trials_data.append({
        "Trial ID": trial.trial_id,
        "Layers": p['num_dense_layers'],
        "Units": p['units'],
        "Dropout": round(p['dropout'], 1),
        "Learn Rate": p['learning_rate'],
        "Val Accuracy (%)": round(trial.score * 100, 2)
    })

print("\n============== DENSE NN HYPERPARAMETER TRIALS LEADERBOARD ==============")
print(pd.DataFrame(trials_data).to_string(index=False))

# Retrieve the #1 best model and evaluate on unseen Test Data
best_hps = tuner.get_best_hyperparameters(1)[0]
best_model = tuner.get_best_models(1)[0]
results = best_model.evaluate(X_test, y_test_oh, verbose=0)

test_loss = results[0]
test_acc = results[1]

print("\n================= BEST DENSE NN CONFIGURATION & RESULT =================")
print(f"Best Dense Layers     : {best_hps.get('num_dense_layers')}")
print(f"Best Hidden Units     : {best_hps.get('units')}")
print(f"Best Dropout Rate     : {best_hps.get('dropout'):.1f}")
print(f"Best Learning Rate    : {best_hps.get('learning_rate')}")
print(f"Unseen Test Accuracy  : {test_acc * 100:.2f}%")
print(f"Unseen Test Loss      : {test_loss:.4f}")
print("========================================================================")


============== DENSE NN HYPERPARAMETER TRIALS LEADERBOARD ==============
Trial ID  Layers  Units  Dropout  Learn Rate  Val Accuracy (%)
       2       1    256      0.4      0.0001             77.77
       5       1    256      0.3      0.0001             77.62
       7       2    128      0.4      0.0005             77.00
       0       2    128      0.3      0.0005             76.68
       1       2    256      0.4      0.0005             76.56
       3       1    256      0.5      0.0005             76.55
       6       1    256      0.3      0.0010             76.21
       4       2    256      0.3      0.0010             76.11


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



================= BEST DENSE NN CONFIGURATION & RESULT =================
Best Dense Layers     : 1
Best Hidden Units     : 256
Best Dropout Rate     : 0.4
Best Learning Rate    : 0.0001
Unseen Test Accuracy  : 77.78%
Unseen Test Loss      : 0.8087
